In [ ]:
import os
from pathlib import Path
import fitsio
from astropy.table import Table
import pandas as pd

from utils import *

In [ ]:
#read in common catalog
all_cat_base_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II")
vi_base_path = Path(os.environ["CFS"]) / "desi" / "users" / "bid13" / "DESI_II" / "pilot_obs" / "DESI-II-VI-results" 

all_cat = Table(fitsio.read(all_cat_base_path / "pilot_obs/MERGED/merged_cat_LSST_WL_Y1_no_VI.fits" ))

### XMM

In [ ]:
xmm_cat = all_cat[all_cat["FIELD_NAME"]=="XMMLSS"]
print(f"There are {len(xmm_cat)} objects in this field")

In [ ]:
xmm_vi_path = vi_base_path / "XMMLSS"

In [ ]:
#read in all files
vi = []
for f in xmm_vi_path.glob("*.csv"):
    vi_single = pd.read_csv(f, delimiter = ",", engine='python', keep_default_na=False)
    vi.append(vi_single)
vi = pd.concat(vi)

vi_gp = vi.groupby(['TARGETID'])
print('There are ' + str(len(vi)) + ' visual inspections of a total of ' + str(len(vi_gp)) + ' unique objects')

#### check if there is any missing VI

In [ ]:
un_resp = np.unique(vi[vi["VI_quality"]<0]["TARGETID"])
print(f"{len(un_resp)} have missing inputs")
missing = xmm_cat["TARGETID"][~np.isin(xmm_cat["TARGETID"], np.unique(vi["TARGETID"]))]
print(f"{len(missing)}  objects are missing")

In [ ]:
# ### Adding useful columns.  Automatically decide on best_z, spectype, and quality where possible.  Concatenate issues and comments.
vi.reset_index(inplace=True)
vi.drop(vi[np.isin(vi["TARGETID"], un_resp)].index, inplace=True)
vi.drop(vi[np.isin(vi["TARGETID"], missing)].index, inplace=True)
choose_best_z(vi)
choose_best_spectype(vi)
choose_best_quality(vi)
concatenate_all_issues(vi)
concatenate_all_comments(vi)
add_extra_details(vi)

In [ ]:
## Make table with average score
merged_quality = vi.groupby("TARGETID")['VI_quality'].mean()
merged_redshift = vi.groupby("TARGETID")['best_z'].mean()

### look at score conflicts

In [ ]:
conflicts = vi.groupby("TARGETID")['VI_quality'].std()

conflicts = conflicts[np.isfinite(conflicts)] #Nan STD because of one VI

conflicts = conflicts[conflicts>0] # all VI agree

In [ ]:
conflicted_cat = vi[np.isin(vi["TARGETID"], conflicts.index)]

In [ ]:
not_clear = conflicted_cat.groupby("TARGETID")['VI_quality'].mean()
not_clear = not_clear[(not_clear<3) & (not_clear>2)]

conflicted_cat = vi[np.isin(vi["TARGETID"], not_clear.index)]

In [ ]:
inspectors = conflicted_cat.groupby("TARGETID")["VI_scanner"].apply(list)
not_jan = ["JAN" not in x for x in list(inspectors)]
not_dey = ["DEY" not in x for x in list(inspectors)]
not_veto = inspectors.index[np.logical_and(not_jan,not_dey)]
conflicted_cat = vi[np.isin(vi["TARGETID"], not_veto)]

In [ ]:
conflicted_targets = np.concatenate([np.unique(conflicted_cat["TARGETID"]), missing, un_resp])

In [ ]:
# np.save(vi_base_path / "conflicted_targets_XMM", conflicted_targets)

### Resolve conflicts and make final table with success indicator

In [ ]:
resolved_conflicts = pd.read_csv(xmm_vi_path / "desi-vi_VI_conflicts_DEY.csv", delimiter = ",", engine='python', keep_default_na=False)
choose_best_z(resolved_conflicts)
resolved_conflicts.set_index(keys="TARGETID", inplace=True)

In [ ]:
resolved_cat = pd.DataFrame(merged_quality[(merged_quality>=3) | (merged_quality<=2)])
resolved_cat = resolved_cat.join(merged_redshift,how="left")

resolved_cat = pd.concat([resolved_conflicts[["VI_quality","best_z"]],resolved_cat])

In [ ]:
#veto DEY and JAN
conflicted_cat = vi[np.isin(vi["TARGETID"], not_clear.index)]

inspectors = conflicted_cat["VI_scanner"]
jan = ["JAN"==x for x in list(inspectors)]
dey = ["DEY"==x for x in list(inspectors)]

jan = conflicted_cat[jan]
jan_cat = vi[np.isin(vi["TARGETID"], np.unique(jan["TARGETID"]))]
jan_cat = jan_cat[jan_cat["VI_scanner"]=="JAN"]
jan_cat.set_index(keys="TARGETID", inplace=True)

dey = conflicted_cat[dey]
dey_cat = vi[np.isin(vi["TARGETID"], np.unique(dey["TARGETID"]))]
dey_cat = dey_cat[dey_cat["VI_scanner"]=="DEY"]
dey_cat.set_index(keys="TARGETID", inplace=True)

#remove the missing values

#add redshift

In [ ]:
resolved_cat = pd.concat([jan_cat[["VI_quality","best_z"]],resolved_cat, dey_cat[["VI_quality","best_z"]]])

In [ ]:
resolved_cat.to_csv(vi_base_path / "merged_vi_resolved_XMMLSS.csv")